# 🏥 Prescription Logic (MED-RT) 

## ── SECTION 0: Environment & Logging ──

In [2]:
import sys, platform, importlib, datetime, logging, os

# ─── Environment  ────────────────────────────────
print("=" * 55)
print("ENVIRONMENT INFO")
print("=" * 55)
print(f"Date/Time : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Python    : {sys.version}")
print(f"Platform  : {platform.platform()}")
for pkg in ["requests", "pandas", "functools"]:
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg:<18} {getattr(m, '__version__', 'stdlib')}")
    except ImportError:
        print(f"  {pkg:<18} NOT INSTALLED")
print("=" * 55)

# ─── Logging setup ───────────────────────────────────────
LOG_FILE = "prescription_logic.log"
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="w"),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger(__name__)
logger.info("Prescription Logic started")
print(f"\nLogging → {LOG_FILE}")

ENVIRONMENT INFO
Date/Time : 2026-04-06 09:24:56
Python    : 3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]
Platform  : Windows-10-10.0.26100-SP0
  requests           2.32.5
  pandas             2.3.3
  functools          stdlib
2026-04-06 09:24:57,811 | INFO     | Prescription Logic started

Logging → prescription_logic.log


## ── SECTION 1: MedRTClient — Single Consolidated API Client ──

In [3]:
import requests
import time
import json
import csv
import functools
from typing import Optional, List, Dict, Any

# ─── Constants ───────────────────────────────────────────
BASE_URL      = "https://rxnav.nlm.nih.gov/REST"
API_TIMEOUT   = 10         
MAX_RETRIES   = 3
RETRY_DELAY   = 1.5        
DISK_CACHE_FILE = "medrt_cache.json"


class MedRTClient:
    """
    Client for the MED-RT / RxNav API.

    Fixes applied:
    - Single consolidated class (was two near-identical standalone functions)
    - Timeout + retry with exponential backoff on every request
    - raise_for_status() before .json()
    - In-memory LRU cache (functools.lru_cache via a dict wrapper)
    - Optional persistent disk cache (JSON file)
    - Structured logging throughout
    """

    def __init__(self, use_disk_cache: bool = True):
        self._mem_cache: Dict[str, List[str]] = {}
        self.use_disk_cache = use_disk_cache
        if use_disk_cache:
            self._load_disk_cache()

    # ── Disk cache helpers ───────────────────────────────
    def _load_disk_cache(self) -> None:
        """Load previously cached results from JSON file."""
        if os.path.exists(DISK_CACHE_FILE):
            try:
                with open(DISK_CACHE_FILE, "r", encoding="utf-8") as f:
                    self._mem_cache.update(json.load(f))
                logger.info(f"Disk cache loaded: {len(self._mem_cache)} entries")
            except Exception as e:
                logger.warning(f"Could not load disk cache: {e}")

    def _save_disk_cache(self) -> None:
        """Persist in-memory cache to JSON file."""
        if not self.use_disk_cache:
            return
        try:
            with open(DISK_CACHE_FILE, "w", encoding="utf-8") as f:
                json.dump(self._mem_cache, f, indent=2)
            logger.debug(f"Disk cache saved ({len(self._mem_cache)} entries)")
        except Exception as e:
            logger.warning(f"Could not save disk cache: {e}")

    # ── HTTP helper ──────────────────────────────────────
    def _get(
        self, url: str, params: Dict[str, Any]
    ) -> Optional[Dict]:
        """
        GET with timeout + retry.

        FIX 🔴 CRITICAL:
        - Added timeout=API_TIMEOUT (was absent → could hang forever)
        - Added retry loop with exponential backoff
        - raise_for_status() before .json() (was absent → 4xx/5xx silently failed)
        """
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = requests.get(
                    url, params=params, timeout=API_TIMEOUT
                )
                response.raise_for_status()      
                return response.json()

            except requests.exceptions.Timeout:
                logger.warning(f"Attempt {attempt}/{MAX_RETRIES}: Timeout — {url}")
            except requests.exceptions.ConnectionError as e:
                logger.warning(f"Attempt {attempt}/{MAX_RETRIES}: Connection error — {e}")
            except requests.exceptions.HTTPError as e:
                logger.error(f"HTTP error: {e}")
                return None   
            except ValueError as e:
                logger.error(f"JSON decode error: {e}")
                return None

            if attempt < MAX_RETRIES:
                sleep_s = RETRY_DELAY * attempt
                logger.info(f"Retrying in {sleep_s:.1f}s ...")
                time.sleep(sleep_s)

        logger.error(f"All {MAX_RETRIES} attempts failed for {url}")
        return None

    # ── Core API methods ─────────────────────────────────
    def _find_class_id(self, disease_name: str) -> Optional[Dict[str, str]]:
        """Return {'classId': ..., 'className': ...} or None."""
        data = self._get(
            f"{BASE_URL}/rxclass/class/byName.json",
            {"className": disease_name, "relaSource": "MEDRT"},
        )
        if data is None:
            return None
        concepts = data.get("rxclassMinConceptList", {}).get(
            "rxclassMinConcept", []
        )
        if not concepts:
            logger.info(f"No MED-RT class found for '{disease_name}'")
            return None
        return {
            "classId":   concepts[0]["classId"],
            "className": concepts[0]["className"],
        }

    def _get_class_members(self, class_id: str) -> List[str]:
        """Return sorted list of drug names for a MED-RT class ID."""
        data = self._get(
            f"{BASE_URL}/rxclass/classMembers.json",
            {
                "classId":    class_id,
                "relaSource": "MEDRT",
                "rela":       "may_treat",
                "trans":      "0",
            },
        )
        if data is None:
            return []
        members = data.get("drugMemberGroup", {}).get("drugMember", [])
        names = sorted({m["minConcept"]["name"] for m in members})
        return names

    # ── Public API ───────────────────────────────────────
    def get_drugs_for_disease(
        self,
        disease_name: str,
        limit: Optional[int] = None,
    ) -> Dict[str, Any]:
        """
        Return all drugs indicated for a disease.

        FIX 🔴 CRITICAL: was two near-identical functions.
        FIX 🟠 HIGH:     limit is now a parameter (default = all results).
        FIX 🟠 HIGH:     results are cached in-memory and on disk.

        Returns:
            {
              'disease_query': str,
              'matched_class': str | None,
              'class_id':      str | None,
              'drugs':         list[str],
              'total_found':   int,
              'returned':      int,
              'cached':        bool,
            }
        """
        cache_key = disease_name.strip().lower()

        # ── Cache hit ────────────────────────────────────
        if cache_key in self._mem_cache:
            cached = self._mem_cache[cache_key]
            drugs  = cached["drugs"]
            shown  = drugs[:limit] if limit else drugs
            logger.info(f"Cache hit: '{disease_name}' → {len(drugs)} drugs")
            return {
                "disease_query": disease_name,
                "matched_class": cached.get("matched_class"),
                "class_id":      cached.get("class_id"),
                "drugs":         shown,
                "total_found":   len(drugs),
                "returned":      len(shown),
                "cached":        True,
            }

        # ── API call ─────────────────────────────────────
        logger.info(f"Querying MED-RT for: '{disease_name}'")
        concept = self._find_class_id(disease_name)

        if concept is None:
            return {
                "disease_query": disease_name,
                "matched_class": None,
                "class_id":      None,
                "drugs":         [],
                "total_found":   0,
                "returned":      0,
                "cached":        False,
            }

        all_drugs = self._get_class_members(concept["classId"])

        # ── Cache result ─────────────────────────────────
        self._mem_cache[cache_key] = {
            "drugs":         all_drugs,
            "matched_class": concept["className"],
            "class_id":      concept["classId"],
        }
        self._save_disk_cache()

        shown = all_drugs[:limit] if limit else all_drugs
        logger.info(f"Found {len(all_drugs)} drugs for '{concept['className']}' "
                    f"(showing {len(shown)})")

        return {
            "disease_query": disease_name,
            "matched_class": concept["className"],
            "class_id":      concept["classId"],
            "drugs":         shown,
            "total_found":   len(all_drugs),
            "returned":      len(shown),
            "cached":        False,
        }

    # ── Export helpers ───────────────────────────────────
    def export_to_json(self, results: List[Dict], filepath: str) -> None:
        """Save a list of query results to JSON."""
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2)
        logger.info(f"Results exported → {filepath}")
        print(f"✅ Exported {len(results)} results → {filepath}")

    def export_to_csv(self, results: List[Dict], filepath: str) -> None:
        """Save a list of query results to CSV (one row per drug)."""
        rows = []
        for res in results:
            for drug in res.get("drugs", []):
                rows.append({
                    "disease_query": res["disease_query"],
                    "matched_class": res.get("matched_class", ""),
                    "class_id":      res.get("class_id", ""),
                    "drug_name":     drug,
                    "total_found":   res["total_found"],
                })
        with open(filepath, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(
                f, fieldnames=["disease_query", "matched_class",
                               "class_id", "drug_name", "total_found"]
            )
            writer.writeheader()
            writer.writerows(rows)
        logger.info(f"CSV exported → {filepath} ({len(rows)} rows)")
        print(f"✅ Exported {len(rows)} drug rows → {filepath}")

    # ── Pretty print ─────────────────────────────────────
    @staticmethod
    def format_result(result: Dict, limit_display: int = 15) -> str:
        """Return a formatted string for a single disease result."""
        lines = ["-" * 50]
        if result["matched_class"]:
            cache_tag = " [cached]" if result["cached"] else ""
            lines.append(
                f"Disease: {result['disease_query']}  →  "
                f"MED-RT: {result['matched_class']} "
                f"(ID: {result['class_id']}){cache_tag}"
            )
            lines.append(
                f"Total drugs: {result['total_found']}  "
                f"(showing {min(result['returned'], limit_display)})"
            )
            for i, drug in enumerate(result["drugs"][:limit_display], 1):
                lines.append(f"  {i:>3}. {drug}")
            if result["returned"] > limit_display:
                lines.append(
                    f"  ... and {result['returned'] - limit_display} more. "
                    "Use export_to_csv() to see all."
                )
        else:
            lines.append(
                f"❌ No MED-RT class found for '{result['disease_query']}'. "
                "Try the formal medical name (e.g. 'Diabetes Mellitus')."
            )
        return "\n".join(lines)


print("MedRTClient class defined.")

MedRTClient class defined.


## ── SECTION 2: Instantiate Client ──

In [4]:
# Instantiate once; the disk cache persists across notebook sessions
client = MedRTClient(use_disk_cache=True)
print("Client ready.")

2026-04-06 09:25:04,119 | INFO     | Disk cache loaded: 15 entries
Client ready.


## ── SECTION 3: Programmatic API Usage ──

In [5]:
# ─── Demo queries ────────────────────────────────────────
DEMO_DISEASES = ["Malaria", "Hypertension", "Diabetes Mellitus", "Asthma"]

all_results = []
for disease in DEMO_DISEASES:
    result = client.get_drugs_for_disease(disease)   # limit=None → all drugs
    all_results.append(result)
    print(client.format_result(result, limit_display=10))
    print()

# Export to both formats
client.export_to_json(all_results, "prescription_results.json")
client.export_to_csv(all_results,  "prescription_results.csv")

2026-04-06 09:25:07,955 | INFO     | Cache hit: 'Malaria' → 32 drugs
--------------------------------------------------
Disease: Malaria  →  MED-RT: Malaria (ID: D008288) [cached]
Total drugs: 32  (showing 10)
    1. artemether
    2. chloroquine
    3. chloroquine hydrochloride
    4. chloroquine phosphate
    5. chloroquine sulfate
    6. halofantrine
    7. halofantrine hydrochloride
    8. hydroxychloroquine
    9. hydroxychloroquine sulfate
   10. lumefantrine
  ... and 22 more. Use export_to_csv() to see all.

2026-04-06 09:25:08,000 | INFO     | Cache hit: 'Hypertension' → 230 drugs
--------------------------------------------------
Disease: Hypertension  →  MED-RT: Hypertension (ID: D006973) [cached]
Total drugs: 230  (showing 10)
    1. Angelica sinensis preparation
    2. acebutolol
    3. acebutolol hydrochloride
    4. aliskiren
    5. aliskiren hemifumarate
    6. alprenolol
    7. alprenolol hydrochloride
    8. alprostadil
    9. ambrisentan
   10. amiloride
  ... and 22

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\logging\__init__.py", line 1103, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 58: character maps to <undefined>
Call stack:
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_a

## ── SECTION 4: Evaluation / Validation ──

In [6]:
# Ground-truth: disease → at least these drugs MUST appear in results.
GROUND_TRUTH: Dict[str, List[str]] = {
    "Malaria":           ["chloroquine", "hydroxychloroquine", "artemether"],
    "Hypertension":      ["amlodipine", "lisinopril", "atenolol"],
    "Diabetes Mellitus": ["metformin", "insulin", "acarbose"],
    "Asthma":            ["albuterol", "montelukast"],
    "Arthritis":         ["ibuprofen", "naproxen", "methotrexate"],
    "Common Cold":       ["guaifenesin", "naproxen"],
}

# Negative cases: these diseases are unlikely to yield results (misspelled/informal)
NEGATIVE_CASES = ["fever", "cold", "cough", "headache"]


def evaluate_client(client_instance: MedRTClient) -> dict:
    """Validate API results against ground-truth drug lists."""
    print("\n" + "=" * 65)
    print("EVALUATION: Ground-truth drug coverage")
    print("=" * 65)

    total_drugs_expected = 0
    total_drugs_found    = 0
    disease_results      = {}

    print(f"\n{'Disease':<22} {'Expected':>9} {'Found':>6} {'Coverage':>9}")
    print("-" * 50)

    for disease, expected_drugs in GROUND_TRUTH.items():
        res          = client_instance.get_drugs_for_disease(disease)
        returned_set = {d.lower() for d in res["drugs"]}
        n_expected   = len(expected_drugs)
        n_found      = sum(
            1 for d in expected_drugs if d.lower() in returned_set
        )
        coverage = n_found / n_expected if n_expected else 0
        total_drugs_expected += n_expected
        total_drugs_found    += n_found

        missing = [d for d in expected_drugs if d.lower() not in returned_set]
        status  = "✓" if n_found == n_expected else f"✗ (missing: {', '.join(missing)})"
        print(f"  {disease:<20} {n_expected:>9} {n_found:>6} {coverage:>8.0%}  {status}")
        disease_results[disease] = {
            "coverage": coverage, "missing": missing
        }

    print("-" * 50)
    overall = total_drugs_found / total_drugs_expected if total_drugs_expected else 0
    print(f"  {'OVERALL':<20} {total_drugs_expected:>9} {total_drugs_found:>6} {overall:>8.0%}")

    # Negative cases — informal names should return empty or very few
    print("\nNegative cases (informal disease names — expect 0 or few results):")
    for neg in NEGATIVE_CASES:
        res = client_instance.get_drugs_for_disease(neg)
        status = "✓ no match" if res["total_found"] == 0 else f"returned {res['total_found']} (MED-RT matched '{res['matched_class']}')"
        print(f"  '{neg}': {status}")

    print("=" * 65)
    logger.info(f"Evaluation complete. Overall coverage: {overall:.1%}")
    return {"overall_coverage": overall, "per_disease": disease_results}


eval_metrics = evaluate_client(client)


EVALUATION: Ground-truth drug coverage

Disease                 Expected  Found  Coverage
--------------------------------------------------
2026-04-06 09:25:15,279 | INFO     | Cache hit: 'Malaria' → 32 drugs
  Malaria                      3      3     100%  ✓
2026-04-06 09:25:15,288 | INFO     | Cache hit: 'Hypertension' → 230 drugs
  Hypertension                 3      3     100%  ✓
2026-04-06 09:25:15,294 | INFO     | Cache hit: 'Diabetes Mellitus' → 141 drugs
  Diabetes Mellitus            3      2      67%  ✗ (missing: insulin)
2026-04-06 09:25:15,300 | INFO     | Cache hit: 'Asthma' → 97 drugs
  Asthma                       2      2     100%  ✓
2026-04-06 09:25:15,304 | INFO     | Cache hit: 'Arthritis' → 175 drugs
  Arthritis                    3      3     100%  ✓
2026-04-06 09:25:15,310 | INFO     | Cache hit: 'Common Cold' → 33 drugs
  Common Cold                  2      2     100%  ✓
--------------------------------------------------
  OVERALL                     16     15

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\logging\__init__.py", line 1103, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 58: character maps to <undefined>
Call stack:
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_a

2026-04-06 09:25:18,496 | INFO     | No MED-RT class found for 'cold'
  'cold': ✓ no match
2026-04-06 09:25:18,496 | INFO     | Cache hit: 'cough' → 44 drugs
  'cough': returned 44 (MED-RT matched 'Cough')
2026-04-06 09:25:18,506 | INFO     | Cache hit: 'headache' → 19 drugs
  'headache': returned 19 (MED-RT matched 'Headache')
2026-04-06 09:25:18,508 | INFO     | Evaluation complete. Overall coverage: 93.8%


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\logging\__init__.py", line 1103, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 56: character maps to <undefined>
Call stack:
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_ai\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Chamikara Pasquel\anaconda3\envs\medical_a

## ── SECTION 5: Interactive Session ──

> **FIX 🟠 HIGH:** Original embedded `while True + input()` directly in the cell,
> which blocks the Jupyter kernel. It is now a standalone function.

In [8]:
def run_interactive(client_instance: MedRTClient) -> None:
    """Interactive disease-to-drug query loop."""
    print("🏥 --- Disease to Drug Lookup (MED-RT) ---")
    print("  Tip: use formal medical names (e.g. 'Diabetes Mellitus').")
    print("  Type 'export' to save session results. Type 'quit' to exit.\n")

    session_results = []

    while True:
        try:
            user_input = input(" Enter a Disease Name: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSession ended.")
            break

        if user_input.lower() in ("quit", "exit", ""):
            print("Goodbye! 👋")
            break

        if user_input.lower() == "export":
            if session_results:
                client_instance.export_to_csv(session_results,  "session_results.csv")
                client_instance.export_to_json(session_results, "session_results.json")
            else:
                print("  Nothing to export yet.")
            continue

        result = client_instance.get_drugs_for_disease(user_input)
        session_results.append(result)
        print(client_instance.format_result(result))
        print("=" * 50 + "\n")


# Uncomment to start the interactive session:
run_interactive(client)

🏥 --- Disease to Drug Lookup (MED-RT) ---
  Tip: use formal medical names (e.g. 'Diabetes Mellitus').
  Type 'export' to save session results. Type 'quit' to exit.



 Enter a Disease Name:  Diabetes


2026-04-06 10:52:15,127 | INFO     | Querying MED-RT for: 'Diabetes'
2026-04-06 10:52:16,948 | INFO     | No MED-RT class found for 'Diabetes'
--------------------------------------------------
❌ No MED-RT class found for 'Diabetes'. Try the formal medical name (e.g. 'Diabetes Mellitus').



 Enter a Disease Name:  Diabetes Mellitus


2026-04-06 10:52:25,970 | INFO     | Cache hit: 'Diabetes Mellitus' → 141 drugs
--------------------------------------------------
Disease: Diabetes Mellitus  →  MED-RT: Diabetes Mellitus (ID: D003920) [cached]
Total drugs: 141  (showing 15)
    1. Capsicum extract
    2. Capsicum oleoresin
    3. acarbose
    4. acetohexamide
    5. aflibercept
    6. aflibercept-ayyh
    7. aflibercept-boav
    8. aflibercept-mrbb
    9. aflibercept-yszy
   10. albiglutide
   11. alogliptin
   12. alogliptin benzoate
   13. becaplermin
   14. benazepril
   15. benazepril hydrochloride
  ... and 126 more. Use export_to_csv() to see all.



...

2026-04-06 11:00:43,196 | INFO     | Querying MED-RT for: 'primary insomnia'
2026-04-06 11:00:44,959 | INFO     | No MED-RT class found for 'primary insomnia'
--------------------------------------------------
❌ No MED-RT class found for 'primary insomnia'. Try the formal medical name (e.g. 'Diabetes Mellitus').



 Enter a Disease Name:   restless leg syndrome


2026-04-06 11:01:03,777 | INFO     | Querying MED-RT for: 'restless leg syndrome'
2026-04-06 11:01:07,077 | INFO     | Found 4 drugs for 'Restless Legs Syndrome' (showing 4)
--------------------------------------------------
Disease: restless leg syndrome  →  MED-RT: Restless Legs Syndrome (ID: D012148)
Total drugs: 4  (showing 4)
    1. carbamazepine
    2. clonazepam
    3. iron carbonyl
    4. rotigotine



 Enter a Disease Name:  exit


Goodbye! 👋
